In [0]:
%sql
-- 1. Aseguramos que la tabla no tenga basura previa
-- DROP TABLE IF EXISTS workspace.products.silver_products;

In [0]:
%sql
DESCRIBE TABLE workspace.products.catalog;

In [0]:
%sql
DESCRIBE TABLE workspace.products.bronze_scraped_products_clean;

In [0]:
%sql
-- ALTER TABLE workspace.products.silver_products SET
-- TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported')

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.products.silver_products (
    product_id STRING,
    retailer STRING,
    brand STRING,
    name STRING,
    list_price DOUBLE,
    cash_price DOUBLE,
    scraped_at TIMESTAMP,
    scraped_date DATE,
    discount_pct DOUBLE,
    installments_json STRING
    -- is_available BOOLEAN DEFAULT TRUE,
    -- is_valid_name BOOLEAN DEFAULT TRUE,
    -- is_valid_price BOOLEAN DEFAULT TRUE,
    -- discount_applied_str STRING DEFAULT "",
    -- discount_applied_int DOUBLE DEFAULT null,
    -- has_installments BOOLEAN DEFAULT TRUE
) 
USING DELTA 
PARTITIONED BY (scraped_date, retailer);

ALTER TABLE workspace.products.silver_products SET
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%sql
DESCRIBE TABLE workspace.products.silver_products;

In [0]:
%sql
MERGE INTO workspace.products.silver_products AS target
USING (
  -- Aquí procesamos los datos de Bronze antes de insertarlos
  SELECT 
    b.product_id,
    b.retailer,
    c.brand, -- Usamos la marca del catálogo que es más confiable
    b.name,
    TRY_CAST(REGEXP_REPLACE(b.list_price, '[\\$.]', '') AS DOUBLE) AS list_price,
    TRY_CAST(REGEXP_REPLACE(b.cash_price, '[\\$.]', '') AS DOUBLE) AS cash_price,
    b.scraped_at,
    CAST(b.scraped_at AS DATE) AS scraped_date,
    b.installments_json
  FROM workspace.products.bronze_scraped_products_clean b
  INNER JOIN workspace.products.catalog c 
    ON b.product_id = c.product_id AND b.retailer = c.retailer
) AS source
ON target.product_id = source.product_id 
   AND target.retailer = source.retailer 
   AND target.scraped_at = source.scraped_at
-- Si ya existe esa combinación exacta, podrías actualizar (opcional)
WHEN MATCHED THEN
  UPDATE SET 
    target.list_price = source.list_price,
    target.cash_price = source.cash_price
-- Si no existe, es un dato nuevo del scraping de hoy y se inserta
WHEN NOT MATCHED THEN
  INSERT (product_id, retailer, brand, name, list_price, cash_price, scraped_at, scraped_date, discount_pct, installments_json)
  VALUES (
    source.product_id, 
    source.retailer, 
    source.brand, 
    source.name, 
    source.list_price, 
    source.cash_price, 
    source.scraped_at, 
    source.scraped_date,
    ROUND((1 - (source.cash_price / source.list_price)) * 100, 2),
    source.installments_json
  )

In [0]:
%sql
-- ALTER TABLE products.silver_products
-- ADD COLUMN is_available BOOLEAN;


UPDATE products.silver_products
SET is_available = CASE
  WHEN list_price IS NULL AND cash_price IS NULL THEN FALSE
  ELSE TRUE
END;

In [0]:
%sql
-- ALTER TABLE products.silver_products
-- ADD COLUMN is_valid_name BOOLEAN;


UPDATE products.silver_products
SET is_valid_name = CASE
  WHEN name IS NULL OR name = '' THEN FALSE
  ELSE TRUE
END;

In [0]:
%sql
-- ALTER TABLE products.silver_products
-- ADD COLUMN is_valid_price BOOLEAN;


UPDATE products.silver_products
SET is_valid_price = CASE
  WHEN list_price IS NULL AND cash_price IS NULL THEN FALSE
  WHEN list_price > 10000000 AND cash_price > 10000000 THEN FALSE
  WHEN list_price < 0 AND cash_price < 0 THEN FALSE
  ELSE TRUE
END;

In [0]:
%sql
-- ALTER TABLE products.silver_products
-- ADD COLUMN discount_applied_str STRING;


UPDATE products.silver_products
SET discount_applied_str = CASE
  WHEN list_price IS NULL OR cash_price IS NULL OR list_price = 0 THEN "0"
  ELSE CONCAT(ROUND((1 - (cash_price / list_price)) * 100,2), "%")
END;

In [0]:
%sql
-- ALTER TABLE products.silver_products
-- ADD COLUMN discount_applied_int DOUBLE;


UPDATE products.silver_products
SET discount_applied_int = CASE
  WHEN list_price IS NULL OR cash_price IS NULL OR list_price = 0 THEN 0.0
  ELSE ROUND((1 - (cash_price / list_price)) * 100, 2)
END;

In [0]:
%sql
-- ALTER TABLE products.silver_products
-- ADD COLUMN has_installments BOOLEAN;


UPDATE products.silver_products
SET has_installments = CASE
    -- Si el campo tiene contenido relevante o menciona cuotas sin interés
    WHEN (LOWER(installments_json) LIKE '%sin interés%' OR LOWER(installments_json) LIKE '%cuotas fijas%') THEN TRUE
    -- Si no tiene texto pero el número extraído es mayor a 1 (Caso Fravega/Megatone)
    WHEN ARRAY_MAX(
           TRANSFORM(
             REGEXP_EXTRACT_ALL(installments_json, ':\\s*"*(\\d+)', 1), 
             x -> CAST(x AS INT)
           )
         ) > 1 THEN TRUE
    ELSE FALSE
  END;